# Notebook 4 – Duplicate Data Handling
**File:** `04_Duplicate_Data.ipynb`

## Learning objectives
Understand exact and partial duplicates, detect them with pandas, remove only records that violate a defined business rule, and document decisions.

**Why preprocessing is required:** Duplicate records can inflate counts, revenue, averages, and model training weight. But two rows that look similar are not necessarily the same real-world event.

## 1. What are duplicate records?
A duplicate is a repeated representation of the same entity or event. Duplicates may arise from repeated form submissions, retrying an API request, importing the same file twice, or joining tables incorrectly.

- **Simple example:** The same student row appears twice.
- **Real-world example:** A customer submits the same registration form twice.
- **Business example:** An order is imported twice, making sales totals too high.
- **AI/ML use case:** Repeated training examples can over-represent some customers or transactions and bias a model.

In [1]:
import pandas as pd

students = pd.DataFrame({
    "StudentID": [101, 102, 103, 103, 104],
    "Name": ["Asha", "Ravi", "Mina", "Mina", "John"],
    "City": ["Hyderabad", "Chennai", "Delhi", "Delhi", "Pune"],
    "Marks": [85, 78, 91, 91, 88]
})
display(students)

,StudentID,Name,City,Marks
0,101,Asha,Hyderabad,85
1,102,Ravi,Chennai,78
2,103,Mina,Delhi,91
3,103,Mina,Delhi,91
4,104,John,Pune,88


## 2. Exact duplicates
An exact duplicate has the same values in every compared column. Check all columns or specify the columns that define a record.

In [2]:
# True for repeated rows after the first occurrence
print(students.duplicated())

# Show duplicate rows (including the first occurrence)
display(students[students.duplicated(keep=False)])

# Count exact duplicate rows after the first occurrence
print("Exact duplicate count:", students.duplicated().sum())

0    False
1    False
2    False
3     True
4    False
dtype: bool


,StudentID,Name,City,Marks
2,103,Mina,Delhi,91
3,103,Mina,Delhi,91


Exact duplicate count: 1


**When useful:** When the dataset's full row represents one unique event and all fields should match.

**Limitation:** If timestamps, formatting, or non-key metadata differ, the same real-world event may not be detected as an exact duplicate.

## 3. Partial duplicates
A partial duplicate matches on selected identity columns, even if other fields differ. For example, the same customer ID may appear with a changed phone number.

There is no universal key: choose columns based on what one row represents.

In [3]:
partial = pd.DataFrame({
    "CustomerID": [1, 2, 1, 3, 2],
    "Name": ["Asha", "Ravi", "Asha", "Mina", "Ravi"],
    "Phone": ["111", "222", "999", "333", "222"],
    "UpdatedAt": ["2026-01-01", "2026-01-02", "2026-02-01", "2026-01-03", "2026-01-02"]
})
display(partial)

# Detect repeated CustomerID values
display(partial[partial.duplicated(subset=["CustomerID"], keep=False)])

,CustomerID,Name,Phone,UpdatedAt
0,1,Asha,111,2026-01-01
1,2,Ravi,222,2026-01-02
2,1,Asha,999,2026-02-01
3,3,Mina,333,2026-01-03
4,2,Ravi,222,2026-01-02


,CustomerID,Name,Phone,UpdatedAt
0,1,Asha,111,2026-01-01
1,2,Ravi,222,2026-01-02
2,1,Asha,999,2026-02-01
4,2,Ravi,222,2026-01-02


**Important:** Two records with the same customer ID may be historical versions, not duplicates. The business rule may be to keep the latest version rather than delete one arbitrarily.

## 4. `duplicated()` options
- `keep='first'`: mark later occurrences as duplicates.
- `keep='last'`: mark earlier occurrences as duplicates.
- `keep=False`: mark every occurrence in a duplicate group as duplicate.

In [4]:
demo = pd.DataFrame({"ID": [1, 2, 2, 3, 3, 3], "Value": ["A", "B", "B", "C", "C", "C"]})
display(demo.assign(keep_first=demo.duplicated(keep="first"),
                    keep_last=demo.duplicated(keep="last"),
                    keep_false=demo.duplicated(keep=False)))

,ID,Value,keep_first,keep_last,keep_false
0,1,A,False,False,False
1,2,B,False,True,True
2,2,B,True,False,True
3,3,C,False,True,True
4,3,C,True,True,True
5,3,C,True,False,True


## 5. `drop_duplicates()`
Use `drop_duplicates()` only after confirming what makes a record unique. It returns a new DataFrame by default; use the returned result or assign it.

In [5]:
# Remove exact duplicate rows, retaining the first
students_no_exact_dupes = students.drop_duplicates()
display(students_no_exact_dupes)

# Remove duplicates based on selected columns
# Here, assume StudentID uniquely identifies a current student record.
students_one_per_id = students.drop_duplicates(subset=["StudentID"], keep="first")
display(students_one_per_id)

,StudentID,Name,City,Marks
0,101,Asha,Hyderabad,85
1,102,Ravi,Chennai,78
2,103,Mina,Delhi,91
4,104,John,Pune,88


,StudentID,Name,City,Marks
0,101,Asha,Hyderabad,85
1,102,Ravi,Chennai,78
2,103,Mina,Delhi,91
4,104,John,Pune,88


## 6. Keep the latest record using a business rule
For customer profiles, the newest update may be the correct record. Sort by the timestamp and retain the latest row per customer.

In [6]:
partial["UpdatedAt"] = pd.to_datetime(partial["UpdatedAt"])

latest_customer = (
    partial.sort_values("UpdatedAt")
           .drop_duplicates(subset=["CustomerID"], keep="last")
           .sort_values("CustomerID")
           .reset_index(drop=True)
)
display(latest_customer)

,CustomerID,Name,Phone,UpdatedAt
0,1,Asha,999,2026-02-01
1,2,Ravi,222,2026-01-02
2,3,Mina,333,2026-01-03


## 7. Business rules for duplicate removal
Before removal, answer:
1. What does one row represent: person, account, order, payment, or event?
2. Which columns form the natural/business key?
3. Can two legitimate events share that key?
4. If values conflict, which source is trusted?
5. Should the first, last, highest-quality, or all records be retained?
6. Should records be merged rather than deleted?
7. Is there an audit trail or source-system ID?

**Real-world example:** Two hospital visits by the same patient are not duplicates if they occurred on different dates.

**Business example:** Two payments with the same amount and customer may be two separate purchases. Use transaction ID and payment reference, not amount alone.

## 8. Why blindly deleting duplicates is dangerous
- It can remove legitimate repeated events (multiple purchases or visits).
- It can erase history when a record was updated over time.
- It can discard the most accurate value if the wrong row is retained.
- It can hide upstream system or join problems.
- It can make audit, compliance, and reconciliation difficult.
- It can change class balance and model training data.

A duplicate flag is evidence to investigate—not an automatic instruction to delete.

## 9. Practical business example: orders

In [7]:
orders = pd.DataFrame({
    "OrderID": ["O-1", "O-2", "O-2", "O-3"],
    "CustomerID": [10, 11, 11, 12],
    "Amount": [500, 800, 800, 300],
    "OrderDate": ["2026-01-01", "2026-01-02", "2026-01-02", "2026-01-03"]
})

print("Rows:", len(orders))
print("Unique order IDs:", orders["OrderID"].nunique())
display(orders[orders.duplicated(subset=["OrderID"], keep=False)])

# Only if OrderID is guaranteed unique by the business system:
orders_clean = orders.drop_duplicates(subset=["OrderID"], keep="first")
display(orders_clean)
print("Revenue before:", orders["Amount"].sum())
print("Revenue after:", orders_clean["Amount"].sum())

Rows: 4
Unique order IDs: 3


,OrderID,CustomerID,Amount,OrderDate
1,O-2,11,800,2026-01-02
2,O-2,11,800,2026-01-02


,OrderID,CustomerID,Amount,OrderDate
0,O-1,10,500,2026-01-01
1,O-2,11,800,2026-01-02
3,O-3,12,300,2026-01-03


Revenue before: 2400
Revenue after: 1600


## 10. AI/ML use case
For churn prediction, repeated customer snapshots may give some customers more influence than others. First decide whether rows are customer snapshots at different dates or accidental duplicates. If they are legitimate snapshots, preserve time and split data by customer/time to avoid leakage. If they are ingestion duplicates, resolve them using stable source IDs.

## 11. Final checklist
- Identify the entity/event represented by each row.
- Check exact duplicates with `duplicated()`.
- Check likely keys with `subset=`.
- Inspect all duplicate groups and conflicting fields.
- Apply a documented business rule.
- Keep an audit copy or log of removed/merged rows.
- Recheck row counts, unique keys, and business totals after cleaning.